# 03 — Perceptron ve LightGBM

**Sahibi:** Öğrenci 2 · **Rubrik:** Model Engineering — Perceptron ve GBDT (20 puanın parçası)

| Bölüm | İçerik | Çıktı |
|---|---|---|
| 1 | Veri | — |
| 2 | Referans çizgisi: DummyClassifier | — |
| 3 | Perceptron: parametre taraması | `reports/perceptron_sweep.csv` |

In [1]:
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

In [2]:
import time
from itertools import product

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import Perceptron
from sklearn.pipeline import Pipeline, make_pipeline

from src.config import REPORTS_DIR, SEED
from src.data import load_split, to_xy
from src.features import make_preprocessor
from src.metrics import evaluate, pick_threshold
from src.plots import set_style

# Single switch for the training data: "clean", "smote" or "ctgan" (val/test are always clean)
VARIANT = "clean"

set_style()
pd.set_option("display.float_format", "{:.4f}".format)

## 1. Veri

Öğrenci 1'in temizlenmiş ve doldurulmuş verisi (`clean`). Model seçimi **val** setinde
yapılır; **test** seti en sona, final modellere ayrılmıştır ve bu bölümde yüklenmez.

In [3]:
train = load_split("train", VARIANT)
val = load_split("val", "clean")
X_train, y_train = to_xy(train)
X_val, y_val = to_xy(val)

pd.DataFrame(
    {
        "rows": [len(train), len(val)],
        "default_rate": [y_train.mean(), y_val.mean()],
    },
    index=[f"train ({VARIANT})", "val"],
)

,rows,default_rate
train (clean),20975,0.2213
val,4495,0.2211


## 2. Referans çizgisi: DummyClassifier

`DummyClassifier` özelliklere hiç bakmaz. Gerçek modellerin skorları ancak bu referansla
kıyaslanınca anlam kazanır.

- **most_frequent:** herkese çoğunluk sınıfını (0 = ödeyecek) söyler.
- **stratified:** sınıf oranına göre rastgele tahmin eder (%22 olasılıkla 1).

In [4]:
results = {}
for strategy in ("most_frequent", "stratified"):
    dummy = DummyClassifier(strategy=strategy, random_state=SEED).fit(X_train, y_train)
    # Dummy models have no meaningful score to tune a threshold on, so predict() is used as is
    y_score = dummy.predict_proba(X_val)[:, 1]
    results[f"dummy_{strategy}"] = evaluate(y_val, y_score, dummy.predict(X_val))

pd.DataFrame(results).T

,macro_f1,weighted_f1,roc_auc,pr_auc,accuracy
dummy_most_frequent,0.4378,0.6820,0.5000,0.2211,0.7789
dummy_stratified,0.5019,0.6573,0.5019,0.2218,0.6578


**Yorum**

- **Accuracy tuzağı:** `most_frequent` tek bir temerrüdü bile yakalamadan **%77,9 accuracy**
  alıyor. Macro-F1 ise 0,438: sınıf 0'ın F1'i ≈ 0,876, sınıf 1'in F1'i 0, ortalaması 0,438.
  Accuracy bu yüzden yalnızca bilgi amaçlı raporlanıyor.
- **Eşikten bağımsız metriklerin tabanı:** rastgele skorlarda ROC-AUC ≈ 0,50 ve PR-AUC ≈ val'deki
  pozitif oranı (0,221). PR eğrisinin taban çizgisi bu yüzden 0,22.
- **`stratified`** sınıf oranında rastgele tahminle macro-F1 ≈ 0,50 alıyor; accuracy'si ise
  çoğunluk tahmininden *düşük* (0,658). Accuracy'ye bakan biri yanlış modeli seçerdi.
- **Hedef:** Gerçek modellerin macro-F1'de 0,50'nin, PR-AUC'de 0,22'nin belirgin üstüne çıkması
  gerekiyor. Öğrenci 1'in sabit parametreli LightGBM'i val'de 0,707 / 0,550 almıştı
  (`reports/tstr_trtr.csv`), iyi bir sonucun ölçeği bu.

## 3. Perceptron

Perceptron skoru $s = w \cdot x + b$ hesaplar ve $s > 0$ ise 1 der. Yalnızca **yanlış
sınıflandırdığı** satırda ağırlıkları günceller: $w \leftarrow w + \eta_0 \, y \, x$
($y \in \{-1, +1\}$).

- **Ön işleme:** `make_preprocessor("linear")` — nominal kolonlara one-hot, geri kalanına
  RobustScaler. Pipeline içinde olduğu için ölçekleyici **yalnızca train'e** fit edilir;
  val'e train'in medyan/IQR değerleriyle uygulanır.
- **Skor ve eşik:** Perceptron olasılık vermez; ham skor `decision_function` kullanılır.
  Eşik val'de `pick_threshold` ile macro-F1'i maksimize edecek şekilde seçilir.
- **Tarama:** `eta0` × `penalty` × `alpha` × `class_weight`. `penalty=None` iken `alpha`
  kullanılmadığı için o kombinasyonlar tek kez eğitilir (50 konfigürasyon).
- **5 tohum:** Veri doğrusal ayrılamadığı için Perceptron'un ağırlıkları salınır ve sonuç
  satırların karıştırılma sırasına (`random_state`) çok duyarlıdır. Tek tohumla seçim yapmak
  şanslı bir sırayı seçmek olur. Bu yüzden her konfigürasyon 5 tohumla eğitilir, seçim
  **ortalama** macro-F1'e göre yapılır, standart sapma da raporlanır.

In [5]:
ETA0S = [1e-4, 1e-3, 1e-2, 0.1, 1.0]
PENALTIES = [None, "l2", "elasticnet"]
ALPHAS = [1e-4, 1e-3]
CLASS_WEIGHTS = [None, "balanced"]
SEEDS = [SEED + i for i in range(5)]
METRICS = ["macro_f1", "weighted_f1", "roc_auc", "pr_auc", "accuracy"]


def make_perceptron(seed: int = SEED, **params) -> Pipeline:
    params = {"max_iter": 1000} | params
    return make_pipeline(make_preprocessor("linear"), Perceptron(random_state=seed, **params))


def score_on_val(model: Pipeline) -> dict:
    """Val metrics at the macro-F1-optimal threshold, which is picked on val itself."""
    y_score = model.decision_function(X_val)
    threshold = pick_threshold(y_val, y_score)
    return evaluate(y_val, y_score, (y_score >= threshold).astype(int))


rows = []
start = time.perf_counter()
for eta0, penalty, alpha, class_weight in product(ETA0S, PENALTIES, ALPHAS, CLASS_WEIGHTS):
    if penalty is None and alpha != ALPHAS[0]:
        continue  # alpha only scales the penalty term
    params = {"eta0": eta0, "penalty": penalty, "alpha": alpha, "class_weight": class_weight}
    for seed in SEEDS:
        model = make_perceptron(seed, **params).fit(X_train, y_train)
        rows.append(params | {"seed": seed, "n_iter": model[-1].n_iter_} | score_on_val(model))
print(f"{len(rows)} model, {time.perf_counter() - start:.1f} sn")

runs = pd.DataFrame(rows)
# None would be dropped by groupby/pivot and lost in the CSV
runs[["penalty", "class_weight"]] = runs[["penalty", "class_weight"]].fillna("none")
stats = runs.groupby(["eta0", "penalty", "alpha", "class_weight"])[METRICS + ["n_iter"]]
sweep = stats.mean().join(stats.std()[["macro_f1", "pr_auc"]], rsuffix="_std").reset_index()
REPORTS_DIR.mkdir(exist_ok=True)
sweep.to_csv(REPORTS_DIR / "perceptron_sweep.csv", index=False)
cols = ["eta0", "penalty", "alpha", "class_weight"]
cols += ["macro_f1", "macro_f1_std", "pr_auc", "pr_auc_std"]
sweep.sort_values("macro_f1", ascending=False)[cols].head(10)

250 model, 105.3 sn


,eta0,penalty,alpha,class_weight,macro_f1,macro_f1_std,pr_auc,pr_auc_std
17,0.0010,l2,0.0010,none,0.6356,0.0303,0.4069,0.0680
22,0.0100,elasticnet,0.0010,balanced,0.6312,0.0324,0.3853,0.0504
25,0.0100,l2,0.0001,none,0.6281,0.0184,0.4032,0.0336
32,0.1000,elasticnet,0.0010,balanced,0.6236,0.0656,0.3806,0.0801
19,0.0010,none,0.0001,none,0.6235,0.0350,0.3798,0.0644
43,1.0000,elasticnet,0.0010,none,0.6211,0.0403,0.3884,0.0620
9,0.0001,none,0.0001,none,0.6197,0.0319,0.3673,0.0673
15,0.0010,l2,0.0001,none,0.6178,0.0253,0.3762,0.0618
37,0.1000,l2,0.0010,none,0.6147,0.0068,0.3663,0.0253
45,1.0000,l2,0.0001,none,0.6147,0.0068,0.3663,0.0253


### 3.1 `eta0` neden tek başına etkisiz?

sklearn ağırlıkları $w = 0$ ile başlatır. Her güncelleme $\eta_0$ ile çarpıldığı için eğitim
sonunda $w = \eta_0 \sum y_i x_i$ olur: $\eta_0$ yalnızca **ölçekler**. Skorun işareti, yani
karar sınırı değişmez. Ceza terimi eklenince her adımda ağırlıklar
$(1 - \eta_0 \alpha)$ ile küçültülür; bu çarpan $\eta_0$'a bağlı olduğu için artık sonuç
gerçekten değişir. Etkili büyüklük $\eta_0 \alpha$ çarpımıdır.

Aşağıdaki tabloda (5 tohum ortalaması, `alpha=1e-4`, `class_weight=None`) `none` satırı da
hafifçe oynuyor. Bunun sebebi teori değil, sklearn'ün **erken durma** kriteri: `tol` mutlak bir
eşik, kayıp ise $\eta_0$ ile ölçeklenir. Küçük $\eta_0$'da iyileşme `tol`'un altında kalır ve
eğitim daha erken durur; farklı epoch'ta durmak farklı ağırlık demektir.

In [6]:
subset = sweep[(sweep.alpha == ALPHAS[0]) & (sweep.class_weight == "none")]
subset.pivot(index="penalty", columns="eta0", values="macro_f1")

eta0,0.0001,0.0010,0.0100,0.1000,1.0000
penalty,,,,,
elasticnet,0.6124,0.5968,0.5650,0.5947,0.6068
l2,0.6107,0.6178,0.6281,0.5853,0.6147
none,0.6197,0.6235,0.6047,0.6047,0.6047


In [7]:
# Stopping epoch per seed for penalty=None: small eta0 stops earlier because tol is absolute
no_penalty = runs[(runs.penalty == "none") & (runs.class_weight == "none")]
no_penalty.pivot(index="seed", columns="eta0", values="n_iter")

eta0,0.0001,0.0010,0.0100,0.1000,1.0000
seed,,,,,
42,6,6,6,6,6
43,6,7,7,7,7
44,6,6,8,8,8
45,6,6,15,15,15
46,6,7,13,13,13


In [8]:
# With a fixed number of epochs (tol=None) the weights divided by eta0 match for every seed
FIXED_EPOCHS = 20
same_direction = []
for seed in SEEDS:
    coefs = [
        make_perceptron(seed, eta0=eta0, tol=None, max_iter=FIXED_EPOCHS)
        .fit(X_train, y_train)[-1]
        .coef_.ravel()
        / eta0
        for eta0 in ETA0S
    ]
    same_direction.append(all(np.allclose(coefs[0], c) for c in coefs))
print(f"Sabit {FIXED_EPOCHS} epoch, w / eta0 tüm eta0 değerlerinde aynı:")
print(dict(zip(SEEDS, same_direction)))

Sabit 20 epoch, w / eta0 tüm eta0 değerlerinde aynı:
{42: True, 43: True, 44: True, 45: True, 46: True}


### 3.2 Diğer parametrelerin etkisi

In [9]:
sweep.groupby(["penalty", "class_weight"])[["macro_f1", "pr_auc"]].max().unstack("class_weight")

macro_f1          pr_auc       
class_weight balanced   none balanced   none
penalty                                     
elasticnet     0.6312 0.6211   0.3853 0.3884
l2             0.6111 0.6356   0.3450 0.4069
none           0.6046 0.6235   0.3317 0.3798

### 3.3 Seçilen Perceptron

In [10]:
best = sweep.loc[sweep.macro_f1.idxmax()]
PERCEPTRON_PARAMS = {
    "eta0": float(best.eta0),
    "penalty": None if best.penalty == "none" else best.penalty,
    "alpha": float(best.alpha),
    "class_weight": None if best.class_weight == "none" else best.class_weight,
}
print("Seçilen parametreler:", PERCEPTRON_PARAMS)
print(f"macro-F1 = {best.macro_f1:.4f} ± {best.macro_f1_std:.4f} (5 tohum)")

results["perceptron (5 tohum ort.)"] = best[METRICS]
pd.DataFrame(results).T.astype(float)

Seçilen parametreler: {'eta0': 0.001, 'penalty': 'l2', 'alpha': 0.001, 'class_weight': None}
macro-F1 = 0.6356 ± 0.0303 (5 tohum)


,macro_f1,weighted_f1,roc_auc,pr_auc,accuracy
dummy_most_frequent,0.4378,0.6820,0.5000,0.2211,0.7789
dummy_stratified,0.5019,0.6573,0.5019,0.2218,0.6578
perceptron (5 tohum ort.),0.6356,0.7560,0.6764,0.4069,0.7635


**Yorum**

- **Seçilen model:** `penalty="l2"`, `eta0=1e-3`, `alpha=1e-3`, `class_weight=None`.
  Val'de macro-F1 **0,636 ± 0,030**, PR-AUC 0,407 ± 0,068 (5 tohum). Referans çizgisinin
  (0,502 / 0,222) belirgin üstünde.
- **İlk 10 konfigürasyon birbirinden ayırt edilemiyor:** ortalamalar 0,615–0,636 arasında,
  standart sapmalar ise 0,02–0,07. Farklar tohum gürültüsünden küçük; yani Perceptron'un
  tavanı parametreden çok modelin kendisiyle sınırlı. Tek tohumla yapılan ilk taramada
  "kazanan" çıkan konfigürasyonun (`elasticnet`, `eta0=1`) 5 tohum ortalaması 0,621'de kaldı.
- **`class_weight="balanced"` tutarlı bir kazanç sağlamıyor.** Eşik zaten val'de macro-F1'e
  göre seçildiği için dengesizlik eşikle telafi ediliyor; sınıf ağırlığı ek bilgi getirmiyor.
  Öğrenci 1'in SMOTE için bulduğu sonuçla aynı.
- **Doğrusal modelin sınırı:** ROC-AUC 0,676. Öğrenci 1'in sabit parametreli LightGBM'i
  aynı val setinde 0,777 almıştı. Aradaki fark, özellikler ile temerrüt arasındaki doğrusal
  olmayan ilişkilerden (ör. PAY_x kodlarının etkisi) geliyor; LightGBM bölümünde bu farkı
  ölçeceğiz.
- Final modelde aynı parametreler `random_state=SEED` ile eğitilecek.